In [18]:
import json 
import pandas as pd
import re

In [19]:
with open(r'C:\Gitprojects\Projects\NLP\dataset.json','r') as f:
    data=json.load(f)

In [20]:
print(f"no.of chats:{len(data['customer_service_chats'])}")

no.of chats:15


In [21]:
data['customer_service_chats']

[{'chat_id': 'CHAT_001',
  'timestamp': '2024-11-15T10:23:45Z',
  'conversation': [{'speaker': 'Customer',
    'message': 'Hi, my name is Priya Sharma. I ordered a phone but received a tablet instead. Order number is ORD-445621.'},
   {'speaker': 'Executive',
    'message': 'Hello Priya, I apologize for the mix-up. May I have your contact details?'},
   {'speaker': 'Customer',
    'message': 'Sure, my phone is +91 97834 56721 and email is priya.sharma2024@gmail.com.'},
   {'speaker': 'Executive',
    'message': "Thank you. We'll arrange a replacement within 3 business days."}]},
 {'chat_id': 'CHAT_002',
  'timestamp': '2024-11-16T14:12:33Z',
  'conversation': [{'speaker': 'Customer',
    'message': 'My broadband connection keeps dropping every hour. Complaint ID: CMP2024-XY9988.'},
   {'speaker': 'Executive',
    'message': 'Sorry for the trouble. Could you provide your customer ID and phone number?'},
   {'speaker': 'Customer',
    'message': 'Customer ID is CUST-88991, phone is 080-4

In [22]:
patterns = {
    "email": r"([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.(?:com|in|org|net|edu|gov|co\.in))(?=\s|$|[.,])",

    "phone": r"(?:\+91[-\s]?)?\d{4,5}[-\s]\d{2,3}[-\s]\d{3,4}"
             r"|(?:\+1[-\s]?)?\(?\d{3}\)?[-\s]?\d{3}[-\s]?\d{4}"
             r"|\b\d{10}\b",

    "order_id": r"\b(?:Order|ORD)[-#\s]*[A-Z0-9]{5,}[-]?[A-Z]?\b",
    "ticket_id": r"\bTKT#?\d{4}[-]?\d{4}\b",
    "complaint_id": r"\bCMP\d{4}[-]?[A-Z]{2}\d{4}\b",
    "customer_id": r"\bCUST[-]?\d{5,6}\b",
    "transaction_id": r"\bTXN[-]?\d{6}[-]?[A-Z]{2}\b",
    "service_request": r"\bSR[-]?\d{4}[-]?[A-Z]{2}\b",
    "subscription_id": r"\bSUB[-]?\d{6}[-]?[A-Z]\b",
    "booking_ref": r"\bBK[-]?\d{5}[-]?[A-Z]\b",
    "policy_number": r"\bPOL[-]?\d{6}\b",
    "claim_number": r"\bCLM[-]?\d{4}[-]?\d{6}\b",
    "reference_number": r"\bREF[-]?\d{6}[-]?[A-Z]\b",
    "return_id": r"\bRTN[-]?\d{6}[-]?[A-Z]\b",
    "cancellation_id": r"\bCANC[-]?\d{4}[-]?[A-Z]\b",
}

In [23]:

def extract_pii(conversation):
    """
    Extract PII with improved name patterns
    """
    full_text = " ".join(msg["message"] for msg in conversation)

    pii = {
        "emails": [],
        "phone": [],
        "names": []
    }

    # Extract emails and phones
    pii["emails"] = list(set(re.findall(patterns["email"], full_text, re.IGNORECASE)))
    pii["phone"] = list(set(re.findall(patterns["phone"], full_text)))

    # Name extraction - ONLY from customer messages
    name_set = set()
    
    # Phrases to exclude from name extraction
    blacklist = [
        "switching providers", 
        "account holder", 
        "customer service",
        "locked out",
        "my account",
        "customer id"
    ]

    for msg in conversation:
        # ONLY process customer messages
        if msg.get("speaker", "").lower() != "customer":
            continue

        text = msg["message"]
        
        # Pattern 1: "my name is John Doe" / "I'm John Doe" / "I am John Doe"
        pattern1 = r"(?:my name is|i'm|i am)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)"
        matches1 = re.findall(pattern1, text, flags=re.IGNORECASE)
        
        # Pattern 2: "Name is John Doe" / "Name: John Doe"
        pattern2 = r"(?:name is|name:)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)"
        matches2 = re.findall(pattern2, text, flags=re.IGNORECASE)
        
        # Pattern 3: "Account holder: John Doe"
        pattern3 = r"(?:account holder:)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)"
        matches3 = re.findall(pattern3, text, flags=re.IGNORECASE)
        
        # Pattern 4: "John Doe, phone..." or "John Doe, email..."
        # This catches: "Robert Chen, phone (650)..."
        pattern4 = r"\b([A-Z][a-z]+\s+[A-Z][a-z]+),\s+(?:phone|email)"
        matches4 = re.findall(pattern4, text)
        
        # Combine all matches
        all_matches = matches1 + matches2 + matches3 + matches4
        
        # Filter and clean
        for name in all_matches:
            clean_name = name.strip()
            
            # Skip if empty
            if not clean_name:
                continue
            
            # Skip if matches blacklist
            if any(blacklisted.lower() in clean_name.lower() for blacklisted in blacklist):
                continue
            
            # Skip if contains numbers (like "emily_w_2024")
            if re.search(r'\d', clean_name):
                continue
            
            # Skip if less than 2 words (need first + last name)
            if len(clean_name.split()) < 2:
                continue
            
            # Add to set
            name_set.add(clean_name)

    pii["names"] = list(name_set)
    return pii

In [24]:
def extract_ids(conversation):
    full_text=' '.join([msg['message'] for msg in conversation])
    ids={}
    id_types =['order_id', 'ticket_id', 'complaint_id', 'customer_id', 
                'transaction_id', 'service_request', 'subscription_id',
                'booking_ref', 'policy_number', 'claim_number', 
                'reference_number', 'return_id']
    for id_type in id_types:
        matches=re.findall(patterns[id_type],full_text,re.IGNORECASE)
        if matches:
            ids[id_type]= list(set(matches))
    return ids

In [25]:
def extract_customer_query(conversation):
    customer_messages = [msg['message']for msg in conversation if msg['speaker']=='Customer']
    if customer_messages:
        return customer_messages[0]
    return ""


In [26]:
def extract_resolution(converstaion):
    executive_messages=[msg['message']for msg in converstaion if msg['speaker']== 'Executive']
    if executive_messages:
        #-1 because the last message has always the resolution
        return executive_messages[-1] 
    return ""


In [27]:
def categorize_query(query):
    query_lower=query.lower()
    categories = {
        'Order Issue': ['order', 'delivery', 'shipment', 'package', 'received'],
        'Refund': ['refund', 'money back', 'return'],
        'Technical Support': ['not working', 'error', 'can\'t access', 'locked out', 'login'],
        'Billing': ['charged', 'payment', 'transaction', 'billing'],
        'Cancellation': ['cancel', 'cancellation', 'unsubscribe'],
        'Account Access': ['account', 'password', 'locked', 'access'],
        'Service Request': ['technician', 'repair', 'service', 'fix'],
        'Complaint': ['complaint', 'issue', 'problem'],
        'Information': ['update', 'change', 'modify', 'information']
    }

    for category, keywords in categories.items():
        if any(keyword in query_lower for keyword in keywords):
            return category
        return 'other'

In [28]:
processed_data = []

for chat in data['customer_service_chats']:
    chat_id = chat['chat_id']
    timestamp = chat['timestamp']
    conversation = chat['conversation']
    
    # Extract all information
    pii = extract_pii(conversation)  # ← Using the NEW function
    ids = extract_ids(conversation)
    query = extract_customer_query(conversation)
    resolution = extract_resolution(conversation)
    category = categorize_query(query)
    
    record = {
        'chat_id': chat_id,
        'timestamp': timestamp,
        'category': category,
        'customer_query': query,
        'resolution': resolution,
        'customer_name': ', '.join(pii['names']) if pii['names'] else '',
        'email': ', '.join(pii['emails']) if pii['emails'] else '',
        'phone': ', '.join(pii['phone']) if pii['phone'] else '',
        'order_id': ', '.join(ids.get('order_id', [])),
        'ticket_id': ', '.join(ids.get('ticket_id', [])),
        # ... rest of IDs
    }
    
    processed_data.append(record)

df = pd.DataFrame(processed_data)

In [29]:
df = pd.DataFrame(processed_data)

print(f"Total chats processed: {len(df)}")
print(df['chat_id'].tolist())

Total chats processed: 15
['CHAT_001', 'CHAT_002', 'CHAT_003', 'CHAT_004', 'CHAT_005', 'CHAT_006', 'CHAT_007', 'CHAT_008', 'CHAT_009', 'CHAT_010', 'CHAT_011', 'CHAT_012', 'CHAT_013', 'CHAT_014', 'CHAT_015']


In [30]:
df

,chat_id,timestamp,category,customer_query,resolution,customer_name,email,phone,order_id,ticket_id
0,CHAT_001,2024-11-15T10:23:45Z,Order Issue,"Hi, my name is Priya Sharma. I ordered a phone...",Thank you. We'll arrange a replacement within ...,Priya Sharma,priya.sharma2024@gmail.com,,"Order number, ORD-445621",
1,CHAT_002,2024-11-16T14:12:33Z,other,My broadband connection keeps dropping every h...,And your registered email address?,,vijay_kumar@hotmail.com,9876-543-210,,
2,CHAT_003,2024-11-17T09:45:12Z,other,"Hello, I need to cancel my subscription. My ac...",Got it. May I know the reason for cancellation?,,sarah.johnson@live.com,+1-415-555-0188,,
3,CHAT_004,2024-11-18T16:30:27Z,other,My AC isn't cooling properly. Service request ...,We'll schedule a technician visit within 24 ho...,Arjun Menon,arjun.menon1985@yahoo.in,"080 2233445, +91-9123-456-789",,
4,CHAT_005,2024-11-19T11:05:56Z,other,I was charged twice for the same transaction. ...,Do you have a ticket number for this issue?,,jessica_brown@protonmail.com,(212) 555-0145,,TKT#8877-3322
5,CHAT_006,2024-11-20T13:22:40Z,Order Issue,My delivery was supposed to arrive yesterday b...,Let me check. Your package is out for delivery...,Amit Patel,amit.patel.delhi@gmail.com,+91-9988-77-6655,ORD-334455-B,
6,CHAT_007,2024-11-21T08:15:19Z,other,I can't access my account after the recent upd...,And your phone number?,,linda_garcia@outlook.com,(310) 555-0167,,
7,CHAT_008,2024-11-22T15:40:52Z,Order Issue,The product I received is defective. I want a ...,Got it. Do you have any photos of the defect?,Sneha Reddy,sneha.r2024@yahoo.co.in,"+91-8877-665-544, 9100-223-344",,
8,CHAT_009,2024-11-23T10:55:31Z,other,I need technical support for my printer. It's ...,Thank you. Have you tried restarting the device?,Robert Chen,robert.chen@fastmail.com,(650) 555-0192,,TKT#9988-7766
9,CHAT_010,2024-11-24T12:18:44Z,other,My flight booking got cancelled but I didn't r...,Was this booking done through our app or website?,,maria.lopez@icloud.com,+1-305-555-0123,,


In [32]:
for idx in [0, 5, 10]:
    if idx < len(df):
        print(f"\n--- CHAT {df.iloc[idx]['chat_id']} ---")
        print(f"Category: {df.iloc[idx]['category']}")
        print(f"Customer: {df.iloc[idx]['customer_name']}")
        print(f"Email: {df.iloc[idx]['email']}")
        print(f"Phone: {df.iloc[idx]['phone']}")
        print(f"Order ID: {df.iloc[idx]['order_id']}")
        print(f"Query: {df.iloc[idx]['customer_query'][:100]}...")
        print(f"Resolution: {df.iloc[idx]['resolution'][:100]}...")



--- CHAT CHAT_001 ---
Category: Order Issue
Customer: Priya Sharma
Email: priya.sharma2024@gmail.com
Phone: 
Order ID: Order number, ORD-445621
Query: Hi, my name is Priya Sharma. I ordered a phone but received a tablet instead. Order number is ORD-44...
Resolution: Thank you. We'll arrange a replacement within 3 business days....

--- CHAT CHAT_006 ---
Category: Order Issue
Customer: Amit Patel
Email: amit.patel.delhi@gmail.com
Phone: +91-9988-77-6655
Order ID: ORD-334455-B
Query: My delivery was supposed to arrive yesterday but I haven't received it. Order ID: ORD-334455-B....
Resolution: Let me check. Your package is out for delivery and should reach you today....

--- CHAT CHAT_011 ---
Category: other
Customer: Deepak Gupta
Email: deepak.gupta.finance@gmail.com
Phone: 9876-543-210
Order ID: 
Query: Hello, my insurance claim was rejected. Claim number: CLM-2024-998877....
Resolution: And your email address?...


In [33]:
# View 1: PII Only
pii_columns = ['chat_id', 'customer_name', 'email', 'phone']
pii_df = df[pii_columns].copy()

In [36]:
# View 2: IDs Only
id_columns = ['chat_id', 'order_id', 'ticket_id']
ids_df = df[id_columns].copy()

In [37]:
# View 3: Query-Resolution Pairs
query_resolution_df = df[['chat_id', 'category', 'customer_query', 'resolution']].copy()

In [38]:
# Export full table
df.to_csv('chat_data_complete.csv', index=False)
print("\n✓ Complete data saved to 'chat_data_complete.csv'")

# Export PII table
pii_df.to_csv('chat_data_pii.csv', index=False)
print("✓ PII data saved to 'chat_data_pii.csv'")

# Export IDs table
ids_df.to_csv('chat_data_ids.csv', index=False)
print("✓ IDs data saved to 'chat_data_ids.csv'")

# Export Query-Resolution pairs
query_resolution_df.to_csv('chat_data_queries.csv', index=False)
print("✓ Query-Resolution data saved to 'chat_data_queries.csv'")

# Export to Excel with multiple sheets
with pd.ExcelWriter('chat_data_analysis.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Complete Data', index=False)
    pii_df.to_excel(writer, sheet_name='PII', index=False)
    ids_df.to_excel(writer, sheet_name='IDs', index=False)
    query_resolution_df.to_excel(writer, sheet_name='Queries', index=False)


✓ Complete data saved to 'chat_data_complete.csv'
✓ PII data saved to 'chat_data_pii.csv'
✓ IDs data saved to 'chat_data_ids.csv'
✓ Query-Resolution data saved to 'chat_data_queries.csv'
